# F1 Strategy Analysis

This notebook analyzes and optimizes pit-stop strategies for F1 races.

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from src.strategy.pit_strategy import optimize_pit_strategy, simulate_race_strategy
from src.models.tyre_deg_model import fit_tyre_degradation
from src.data_loader import load_race_data
from src.feature_engineering import add_driver_normalization

# Load Monza data for strategy analysis
df = load_race_data(2023, 'Monza')
df = add_driver_normalization(df)

# Fit degradation model for MEDIUM compound
coeffs_medium = fit_tyre_degradation(df, "MEDIUM")
print(f"Degradation coefficients for MEDIUM: {coeffs_medium}")

# Define degradation function
def degradation_model(lap, compound):
    if compound == "MEDIUM" and coeffs_medium is not None:
        return coeffs_medium[0] * lap**2 + coeffs_medium[1] * lap + coeffs_medium[2]
    else:
        # Default degradation
        return 0.1 * lap  # Simple linear degradation

# Test pit strategy optimization
current_lap = 10
total_laps = 53  # Monza race distance
degradation_rate = 0.05  # seconds per lap

strategy = optimize_pit_strategy(
    current_lap=current_lap,
    total_laps=total_laps,
    degradation_rate=degradation_rate
)

print("Optimized Pit Strategy:")
print(f"Pit laps: {strategy['pits']}")
print(f"Compounds: {strategy['compounds']}")
print(f"Total pit time loss: {strategy['total_time']} seconds")

In [ ]:
# Simulate the race with the optimized strategy
lap_times = simulate_race_strategy(
    total_laps=53,
    pit_strategy=strategy,
    degradation_model=degradation_model,
    base_lap_time=85.0
)

print(f"Total race time: {sum(lap_times):.2f} seconds")
print(f"Average lap time: {np.mean(lap_times):.3f} seconds")

# Plot the race simulation
plt.figure(figsize=(12, 6))
plt.plot(range(1, 54), lap_times, 'b-', linewidth=2, label='Lap Times')
plt.axhline(y=np.mean(lap_times), color='r', linestyle='--', label=f'Average: {np.mean(lap_times):.2f}s')

# Mark pit stops
for pit_lap in strategy['pits']:
    plt.axvline(x=pit_lap, color='orange', linestyle='--', alpha=0.7, label='Pit Stop' if pit_lap == strategy['pits'][0] else "")

plt.xlabel('Lap Number')
plt.ylabel('Lap Time (seconds)')
plt.title('F1 Race Strategy Simulation - Monza 2023')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Compare different strategies
strategies = [
    {"name": "One Stop", "pits": [25], "compounds": ["MEDIUM", "SOFT"]},
    {"name": "Two Stops", "pits": [15, 35], "compounds": ["MEDIUM", "SOFT", "SOFT"]},
    {"name": "No Stops", "pits": [], "compounds": ["MEDIUM"]}
]

strategy_times = []
for strat in strategies:
    times = simulate_race_strategy(53, strat, degradation_model, 85.0)
    total_time = sum(times)
    strategy_times.append((strat["name"], total_time))
    print(f"{strat['name']}: {total_time:.2f} seconds")

# Plot strategy comparison
names, times = zip(*strategy_times)
plt.figure(figsize=(8, 5))
bars = plt.bar(names, times, color=['blue', 'green', 'red'])
plt.ylabel('Total Race Time (seconds)')
plt.title('Pit Strategy Comparison')
plt.grid(True, alpha=0.3)

# Add value labels on bars
for bar, time in zip(bars, times):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{time:.1f}s', ha='center', va='bottom')

plt.show()